In [ ]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

# 1. Initialize the Blackjack Environment
env = gym.make('Blackjack-v1', sab=False)

# 2. Define the Sample Policy
def sample_policy(state):
    """Stick if score >= 20, else Hit."""
    player_sum, _, _ = state
    return 0 if player_sum >= 20 else 1

# 3. Generate an Episode
def generate_episode(env, policy):
    episode = []
    state, info = env.reset()
    while True:
        action = policy(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if terminated or truncated:
            break
    return episode

# 4. Implement First-Visit MC Prediction
def mc_prediction_first_visit(env, num_episodes, policy):
    # Dictionaries to store returns and counts
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)
    V = defaultdict(float)

    for i in range(num_episodes):
        episode = generate_episode(env, policy)

        # Track visited states in this episode to ensure "First-Visit"
        visited_states = set()

        for t, (state, action, reward) in enumerate(episode):
            if state not in visited_states:
                visited_states.add(state)

                # Calculate return G (discounted sum of rewards)
                # For Blackjack, G is just the final reward
                G = reward

                returns_sum[state] += G
                returns_count[state] += 1.0
                V[state] = returns_sum[state] / returns_count[state]

    return V

# Run the simulation
num_episodes = 500000
value_table = mc_prediction_first_visit(env, num_episodes, sample_policy)

# Print some values for inspection
print("Sample State Values (Player Sum, Dealer Card, Usable Ace): Value")
for i, (state, value) in enumerate(list(value_table.items())[:10]):
    print(f"{state}: {value:.4f}")

In [ ]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

# 1. Setup the environment
env = gym.make('Blackjack-v1', sab=False)

# 2. Define the sample policy (Stick if score >= 20, else Hit)
def sample_policy(state):
    player_sum, dealer_card, usable_ace = state
    return 0 if player_sum >= 20 else 1

# 3. First-Visit MC Prediction Function
def first_visit_mc_prediction(env, num_episodes, policy):
    # Dictionaries to track the sum of returns and visit counts for every state
    returns_sum = defaultdict(float)
    N = defaultdict(float)  # Number of visits
    V = defaultdict(float)  # Value function

    for i in range(num_episodes):
        # Generate an episode
        episode = []
        state, info = env.reset()
        while True:
            action = policy(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            episode.append((state, reward))
            state = next_state
            if terminated or truncated:
                break

        # First-Visit MC logic
        visited_states = set()
        for t, (state, reward) in enumerate(episode):
            if state not in visited_states:
                visited_states.add(state)
                # In Blackjack, G is the reward received at the end of the episode
                returns_sum[state] += reward
                N[state] += 1.0
                V[state] = returns_sum[state] / N[state]

    return V

# 4. Run the Monte Carlo Prediction (500,000 episodes)
print("Simulating 500,000 episodes...")
value_table = first_visit_mc_prediction(env, 50000, sample_policy)
print("Simulation complete.\n")

# 5. Print Some Values to inspect
print(f"{'State (Sum, Dealer, Ace)':<25} | {'Value'}")
print("-" * 35)

# Sorting keys for cleaner display
sorted_states = sorted(value_table.items(), key=lambda x: x[0][0], reverse=True)

for state, value in sorted_states[:10]:
    print(f"{str(state):<25} | {value:.4f}")

In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd
from collections import defaultdict

# 1. Initialize the Blackjack Environment
env = gym.make('Blackjack-v1', sab=False)

# Initialize Q-table, return sums, and visit counts
# Using a dictionary to store Q[state][action]
Q = defaultdict(lambda: np.zeros(env.action_space.n))
returns_sum = defaultdict(float)
N = defaultdict(float)

# 2. Define the Epsilon-Greedy Policy
def epsilon_greedy_policy(state, Q, epsilon, nA):
    """
    nA: Number of actions (2: Hit or Stand)
    Returns an action probabilities array
    """
    A = np.ones(nA, dtype=float) * (epsilon / nA)
    best_action = np.argmax(Q[state])
    A[best_action] += (1.0 - epsilon)
    return A

# 3. Generate an Episode
def generate_episode(env, Q, epsilon, nA):
    episode = []
    state, info = env.reset()
    while True:
        # Choose action based on epsilon-greedy policy
        probs = epsilon_greedy_policy(state, Q, epsilon, nA)
        action = np.random.choice(np.arange(nA), p=probs)

        next_state, reward, terminated, truncated, info = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if terminated or truncated:
            break
    return episode

# 4. Run the Monte Carlo Simulation
def mc_control_on_policy(env, num_episodes, epsilon=0.1):
    nA = env.action_space.n

    for i in range(num_episodes):
        episode = generate_episode(env, Q, epsilon, nA)

        # Track first visits to update Q-values
        visited_sa = set()
        for t, (state, action, reward) in enumerate(episode):
            sa_pair = (state, action)
            if sa_pair not in visited_sa:
                visited_sa.add(sa_pair)

                returns_sum[sa_pair] += reward
                N[sa_pair] += 1.0
                Q[state][action] = returns_sum[sa_pair] / N[sa_pair]

    return Q

# Execute simulation
print("Running 50,000 episodes for MC Control...")
Q = mc_control_on_policy(env, 50000)
print("Training complete.\n")

# 5. Store and Display Q-values
# Convert Q-table to a DataFrame
data = []
for state, actions in Q.items():
    data.append({'State (Sum, Dealer, Ace)': state, 'Stand (Q)': actions[0], 'Hit (Q)': actions[1]})

df = pd.DataFrame(data)
print("First 5 rows of the Q-value Table:")
print(df.head())